# Frontieres-06 — Learning to branch
## Auditer la généralisation avant d'annoncer un gain

[← Partie 5](README.md) | [↑ Search](../README.md) | [<< App-25 Enchères WDP/VCG](Frontieres-04-CombinatorialAuctions-WDP-VCG-Python.ipynb)

> **Durée estimée : 75 minutes**

## Hommage et question scientifique

Le groupe **G4 — Simon Naulet et Matis Codjia** a relié deux mondes difficiles à faire dialoguer : la programmation par contraintes et l'apprentissage supervisé. Leur projet construit un mini-solveur CSP, implémente AC-3, extrait des features à chaque nœud et entraîne XGBoost à imiter `dom/wdeg`. Ce passage d'une heuristique écrite à une politique apprise est le geste central conservé ici. Source : [PR PrCon #46](https://github.com/jsboigeEpita/2026-Epita-Programmation-par-Contraintes/pull/46), fusion `9f91222f`.

Une accuracy élevée sur des lignes candidates ne suffit pourtant pas à conclure qu'un solveur est meilleur. Une politique de branchement agit dans une boucle : son choix modifie la propagation, donc les domaines, donc les nœuds futurs qu'elle devra elle-même traiter. Elle peut bien imiter l'oracle localement tout en construisant un arbre plus grand ; elle peut réduire les nœuds tout en perdre en temps si l'inférence coûte plus cher que la décision classique.

La reproduction CoursIA ne copie aucune cellule, fonction, donnée, figure ou prose étudiante. Elle reconstruit indépendamment le dispositif avec `HistGradientBoostingClassifier` et pose une question plus stricte : **la politique apprise généralise-t-elle à de nouvelles instances et à une famille CSP entièrement absente du train, une fois son coût d'inférence compté ?**

### Quatre niveaux à ne pas confondre

1. **Imitation locale** — le candidat classé premier est-il celui choisi par l'oracle ?
2. **Trajectoire de recherche** — la politique ouvre-t-elle moins de nœuds ?
3. **Coût d'exécution** — le gain éventuel amortit-il le scoring à chaque nœud ?
4. **Transfert** — le comportement survit-il quand une famille entière manque au train ?

### Objectifs d'apprentissage

À la fin du parcours, vous saurez construire un split groupé par instance, dériver les features locales et `dom/wdeg`, choisir une baseline sans regarder le test, intégrer une politique apprise dans le solveur, mesurer nœuds/temps/inférence, valider chaque solution et interpréter un résultat négatif sans le diluer.

### Prérequis

- [CSP-6 Hybridation](../Part2-CSP/CSP-6-Hybridization.ipynb)
- [MGS-16 Sélection d'algorithmes](../Part4-Metaheuristics/MGS-16-AlgorithmSelection.ipynb)
- C# et .NET Interactive ; Python 3.13 avec scikit-learn et NumPy via Python.NET 3.1. Les 36 fixtures sont des entrées produites par les générateurs Python de référence.

## 1. Pourquoi le split naïf est trompeur

À un nœud de recherche $n$, chaque variable candidate $x\in C_n$ produit une ligne de features et un label binaire indiquant si `dom/wdeg` l'a choisie. Un split aléatoire de ces **lignes** peut placer un candidat du même nœud dans le train et un autre dans le test. Même si les nœuds sont séparés, deux nœuds issus du même CSP partagent sa taille, sa densité, ses domaines initiaux et sa structure de contraintes.

Cette dépendance viole l'unité d'échantillonnage réelle. Le solveur sera déployé sur une nouvelle instance, pas sur une nouvelle ligne d'une instance déjà vue. Le groupe de split doit donc être `Id`, et non `candidate` ou `Node` seulement.

L'accuracy binaire est elle aussi trompeuse : un nœud avec dix candidats contient neuf labels zéro. Prédire toujours zéro atteint 90 % sans sélectionner de variable. La métrique locale pertinente est le **top-1 par nœud** :

$$\operatorname{top1}=\frac{1}{|N|}\sum_{n\in N}\mathbf 1\!\left[\arg\max_{x\in C_n}s(x)=x_n^*\right].$$

Mais cette fidélité à l'oracle reste intermédiaire. Nous l'accompagnons de la taille de l'arbre, du temps mur, de la part d'inférence et de la validité de l'affectation finale.

Le protocole utilise d'abord un split groupé de 24 instances train / 12 test, puis trois évaluations **leave-one-family-out** : reines, coloration et carrés latins sont successivement absents du train. Cette seconde échelle teste un changement de distribution plus sévère que le simple renouvellement d'instances.

## Exercices — détecter une fuite

Les exercices restent volontairement stubbés afin que le notebook s'exécute de bout en bout. Chacun rejoue un garde méthodologique central plutôt qu'une simple variation syntaxique.

### Exercice 1 — Détecter une fuite par nœud et par instance

Construisez les ensembles de `Node` et d'`Id` des deux côtés. Vérifiez les deux intersections avant d'entraîner.

**Questions :** pourquoi une intersection vide de nœuds ne suffit-elle pas ? Quel objet doit être passé à `un regroupement par instance` ? Comment testeriez-vous la disjonction dans une CI ?

**Étapes suggérées :** filtrer la trace par listes d'instances, extraire les identifiants, calculer les intersections et retourner un dictionnaire de deux booléens. Le stub doit rester sûr si les ensembles sont vides.

In [1]:
// Exercice 1 : construire les ensembles d'instances et nœuds des deux côtés.
var trainNodeIds = new HashSet<string>();
var testNodeIds = new HashSet<string>();
bool? leakage = null; // TODO étudiant
Console.WriteLine(leakage?.ToString() ?? "Exercice à compléter");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Exercice à compléter


### Préparer le modèle

Le pont suivant charge scikit-learn ; les décisions, la recherche et les mesures restent en C#.

In [2]:
// Configurer le pont sklearn : seul le modèle est exécuté par Python.NET.
#r "nuget: pythonnet, 3.1.0"
#load "../../Config/PythonNetLoader.cs"
using Python.Runtime;
using System;
using System.Globalization;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
PythonNetLoader.Initialize(typeof(PythonEngine).Assembly, "sklearn");
using (Py.GIL())
{
    dynamic sklearn = Py.Import("sklearn");
    Console.WriteLine($"scikit-learn : {sklearn.__version__} ; Python.NET initialisé");
}

Installed Packages pythonnet, 3.1.0

scikit-learn : 1.9.1 ; Python.NET initialisé


## 2. Exemple guide — un solveur et un validateur indépendants

Le noyau expérimental utilise des domaines finis, des contraintes binaires, AC-3 et un backtracking chronologique. À chaque affectation tentée, AC-3 révise les arcs voisins jusqu'au point fixe ou jusqu'à vider un domaine. La politique de branchement ne choisit que la **variable** ; les valeurs restent essayées dans l'ordre croissant afin de ne pas confondre deux décisions.

Cette séparation permet une comparaison contrôlée entre quatre baselines et une politique apprise. Le compteur `nodes` augmente à chaque appel de recherche, avant le choix de variable. `inference_seconds` mesure uniquement le temps passé dans la fonction apprise ; le temps total inclut aussi propagation, copie des domaines et récursion.

Le validateur final ne fait confiance ni au statut, ni au nombre de nœuds, ni au chemin de recherche. À partir de l'affectation seule, il vérifie :

1. qu'une valeur est fournie pour chaque variable et seulement pour elles ;
2. que chaque valeur appartient au domaine initial ;
3. que chaque relation binaire est satisfaite dans les deux orientations enregistrées.

Une solution rapide mais incomplète n'est donc pas comptée comme un gain. Ce contrat est essentiel lorsqu'on compare des politiques susceptibles de modifier profondément la trajectoire ou d'atteindre la limite de nœuds.

In [3]:
// Mini-solveur CSP : contraintes binaires, AC-3, heuristiques et validation indépendante.
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Text.Json;
using Python.Runtime;

record Instance(string Family, string Id, Dictionary<string, HashSet<int>> Domains, List<(string, string)> Edges)
{
    public string[] Variables => Domains.Keys.ToArray();

    public IEnumerable<string> Neighbors(string x) => Edges.Where(e => e.Item1 == x || e.Item2 == x).Select(e => e.Item1 == x ? e.Item2 : e.Item1);
    public bool Relation(string x, int a, string y, int b) => a != b && (Family != "queens" || Math.Abs(a - b) != Math.Abs(int.Parse(x[1..]) - int.Parse(y[1..])));
}

record TraceRow(string Family, string Id, string Node, string Candidate, double[] Features, int Label);
record SolveResult(string Family, string Id, string Heuristic, bool Solved, bool Valid, int Nodes, double Seconds, double InferenceSeconds);
class CspSolver
{
    public Instance Csp;
    public string Heuristic;
    public Func<string[], double[][], string> Policy;
    public bool Collect;
    public int Nodes;
    public double Inference;
    public List<TraceRow> Trace = new();
    Dictionary<(string, string), int> weights = new();
    Dictionary<string, int> activity = new();
    public CspSolver(Instance c, string heuristic, Func<string[], double[][], string> policy = null, bool collect = false)
    {
        Csp = c;
        Heuristic = heuristic;
        Policy = policy;
        Collect = collect;
    }

    static (string, string) Edge(string a, string b) => string.CompareOrdinal(a, b) < 0 ? (a, b) : (b, a);
    bool Ac3(Dictionary<string, HashSet<int>> d, IEnumerable<(string, string)> arcs, out HashSet<string> changed)
    {
        var q = new Queue<(string, string)>(arcs);
        changed = new();
        while (q.Count > 0)
        {
            var(x, y) = q.Dequeue();
            var remove = d[x].Where(a => !d[y].Any(b => Csp.Relation(x, a, y, b))).ToArray();
            if (remove.Length == 0)
                continue;
            d[x].ExceptWith(remove);
            changed.Add(x);
            if (d[x].Count == 0)
                return false;
            foreach (var z in Csp.Neighbors(x).Where(z => z != y))
                q.Enqueue((z, x));
        }

        return true;
    }

    double[] Features(string v, Dictionary<string, HashSet<int>> d, Dictionary<string, int> assignment)
    {
        var remain = Csp.Neighbors(v).Where(n => !assignment.ContainsKey(n)).ToArray();
        var sizes = remain.Select(n => d[n].Count).ToArray();
        int wd = remain.Sum(n => weights.GetValueOrDefault(Edge(v, n), 1));
        return new double[]
        {
            d[v].Count,
            (double)d[v].Count / Math.Max(Csp.Domains[v].Count, 1),
            remain.Length,
            (double)remain.Length / Math.Max(Csp.Domains.Count - 1, 1),
            wd,
            activity.GetValueOrDefault(v),
            (double)d[v].Count / Math.Max(wd, 1),
            (double)assignment.Count / Math.Max(Csp.Domains.Count, 1),
            sizes.Length == 0 ? 0 : sizes.Min(),
            sizes.Length == 0 ? 0 : sizes.Average(),
            sizes.Length == 0 ? 0 : sizes.Max()
        };
    }

    string Select(Dictionary<string, HashSet<int>> d, Dictionary<string, int> ass)
    {
        var cand = Csp.Variables.Where(v => !ass.ContainsKey(v)).ToArray();
        var rows = cand.Select(v => Features(v, d, ass)).ToArray();
        string chosen;
        if (Policy != null)
        {
            var sw = Stopwatch.StartNew();
            chosen = Policy(cand, rows);
            sw.Stop();
            Inference += sw.Elapsed.TotalSeconds;
        }
        else
            chosen = Heuristic switch
            {
                "mrv" => cand.OrderBy(v => d[v].Count).ThenBy(v => v, StringComparer.Ordinal).First(),
                "max_degree" => cand.OrderByDescending(v => Csp.Neighbors(v).Count(n => !ass.ContainsKey(n))).ThenBy(v => v, StringComparer.Ordinal).First(),
                "dom_wdeg" => cand.OrderBy(v => Features(v, d, ass)[6]).ThenBy(v => v, StringComparer.Ordinal).First(),
                "activity" => cand.OrderByDescending(v => activity.GetValueOrDefault(v)).ThenBy(v => v, StringComparer.Ordinal).First(),
                _ => throw new ArgumentException(Heuristic)};
        if (Collect && cand.Length > 1)
            for (int i = 0; i < cand.Length; i++)
                Trace.Add(new(Csp.Family, Csp.Id, $"{Csp.Id}:{Nodes}", cand[i], rows[i], cand[i] == chosen ? 1 : 0));
        return chosen;
    }

    Dictionary<string, int> Search(Dictionary<string, int> a, Dictionary<string, HashSet<int>> d)
    {
        if (a.Count == Csp.Domains.Count)
            return new(a);
        if (Nodes >= 20000)
            return null;
        Nodes++;
        var v = Select(d, a);
        foreach (int value in d[v].OrderBy(x => x))
        {
            var next = d.ToDictionary(e => e.Key, e => new HashSet<int>(e.Value));
            var ass = new Dictionary<string, int>(a)
            {
                {
                    v,
                    value
                }
            };
            next[v] = new()
            {
                value
            };
            bool ok = Ac3(next, Csp.Neighbors(v).Select(n => (n, v)), out var changed);
            foreach (var x in changed)
                activity[x] = activity.GetValueOrDefault(x) + 1;
            if (ok)
            {
                var result = Search(ass, next);
                if (result != null)
                    return result;
            }

            foreach (var x in Csp.Neighbors(v))
            {
                var edge = Edge(v, x);
                weights[edge] = weights.GetValueOrDefault(edge, 1) + 1;
            }
        }

        return null;
    }

    public Dictionary<string, int> Solve()
    {
        var d = Csp.Domains.ToDictionary(e => e.Key, e => new HashSet<int>(e.Value));
        var arcs = Csp.Edges.SelectMany(e => new[] { e, (e.Item2, e.Item1) });
        return Ac3(d, arcs, out _) ? Search(new(), d) : null;
    }

    public static bool Validate(Instance c, Dictionary<string, int> solution) => solution != null && solution.Count == c.Domains.Count && c.Domains.All(e => solution.TryGetValue(e.Key, out var v) && e.Value.Contains(v)) && c.Edges.All(e => c.Relation(e.Item1, solution[e.Item1], e.Item2, solution[e.Item2]) && c.Relation(e.Item2, solution[e.Item2], e.Item1, solution[e.Item1]));
    public SolveResult Run()
    {
        var sw = Stopwatch.StartNew();
        var sol = Solve();
        sw.Stop();
        return new(Csp.Family, Csp.Id, Heuristic, sol != null, Validate(Csp, sol), Nodes, sw.Elapsed.TotalSeconds, Inference);
    }
}

Console.WriteLine("Solveur CSP, AC-3 et validateur définis.");

Solveur CSP, AC-3 et validateur définis.


### Le noyau sépare décision, propagation et preuve de validité

La cellule définit un solveur réellement exécuté, pas un replay de métriques. `select` construit les features au nœud courant ; `_search` applique le choix, propage puis revient en arrière ; `validate_solution` sera appelé après la résolution. Une politique apprise doit donc subir ses propres conséquences : un mauvais choix crée les nœuds futurs sur lesquels elle sera ensuite évaluée.

`FEASIBLE` n'est pas un statut d'optimalité ici, car le problème cherche seulement une première solution. Il signifie « affectation complète passée au validateur ». Les comparaisons portent sur les ressources dépensées pour atteindre cette solution, sous la même limite de 20 000 nœuds.

## 3. Instances déterministes et distinctes

Les 36 CSP sont chargés depuis des fixtures d’entrées produites sans réseau : douze tailles de N-reines, douze colorations plantées et douze carrés latins partiellement révélés. Chaque famille impose une géométrie différente à la recherche : contraintes diagonales denses pour les reines, graphe irrégulier pour la coloration, cliques de lignes et colonnes pour les carrés latins.

Chaque `Id` identifie une unité indivisible du split. Les graines de génération sont déterministes, mais cette reproductibilité ne transforme pas 36 problèmes en population représentative des CSP industriels. Le banc sert à éprouver le protocole et le transfert, pas à annoncer une supériorité universelle.

In [4]:
// Les fixtures sont des entrées générées par les générateurs Python, sans solutions.
var input = JsonDocument.Parse(File.ReadAllText("data/frontieres-learned-inputs/csp-instances.json")).RootElement;
var instances = new List<Instance>();
foreach (var e in input.GetProperty("instances").EnumerateArray())
{
    var domains = e.GetProperty("domains").EnumerateObject().ToDictionary(x => x.Name, x => x.Value.EnumerateArray().Select(y => y.GetInt32()).ToHashSet());
    var edges = e.GetProperty("edges").EnumerateArray().Select(x => (x[0].GetString(), x[1].GetString())).ToList();
    instances.Add(new Instance(e.GetProperty("family").GetString(), e.GetProperty("id").GetString(), domains, edges));
}

Console.WriteLine($"Instances : {instances.Count}, familles : {string.Join(", ", instances.GroupBy(x => x.Family).Select(g => $"{g.Key}={g.Count()}"))}");

Instances : 36, familles : queens=12, coloring=12, latin=12


### Trois familles contrôlées, trois formes de décalage

Les N-reines font croître la taille de 8 à 19 variables avec des contraintes de colonne et de diagonale entre chaque paire. Les colorations utilisent 16 à 27 sommets, trois ou quatre couleurs et une densité variable, tout en conservant une coloration plantée. Les carrés latins alternent des ordres 4 à 7 et des taux de révélation différents.

Le générateur garantit la rejouabilité et l'existence d'une solution pour coloration/latin ; il ne garantit pas que toutes les instances aient la même difficulté. C'est pourquoi la baseline est comparée instance par instance avant agrégation, et pourquoi médiane et moyenne géométrique des ratios répondent à des questions complémentaires.

## 4. Dériver `dom/wdeg` et les features locales

Pour une variable non affectée $x$, le degré pondéré courant additionne les poids des contraintes qui la relient à d'autres variables non affectées :

$$wdeg(x)=\sum_{y\in N(x),\,y\notin A}w_{xy}.$$

La règle `dom/wdeg` choisit le plus petit ratio

$$\frac{|D(x)|}{\max(wdeg(x),1)}.$$

Elle favorise une variable au petit domaine impliquée dans des contraintes qui ont déjà contribué à des échecs. Après une branche infructueuse, le prototype incrémente les poids des arêtes incidentes ; `activity` compte séparément les variables dont le domaine a été révisé par AC-3.

Les onze features décrivent quatre dimensions : taille relative du domaine, degré résiduel, mémoire des conflits (`weighted_degree`, `activity`, `dom_wdeg`) et état du voisinage. `progress` situe le nœud dans l'arbre. Elles sont **locales** : elles ne voient ni coût contrefactuel du choix, ni taille de sous-arbre future, ni identité brute de l'instance.

La cible reste l'imitation pointwise de `dom/wdeg`, mais l'évaluation est structurée par nœud. `HistGradientBoostingClassifier` évite une dépendance XGBoost supplémentaire. Pour chaque split, la baseline déployable est choisie par médiane des nœuds **sur le train seulement** parmi MRV, max-degree, `dom/wdeg` et activity.

In [5]:
// Le pont exécute uniquement la construction, l'entraînement et l'inférence sklearn.
class SklearnPolicy : IDisposable
{
    readonly PyObject model;
    readonly PyObject np;
    public SklearnPolicy(IEnumerable<TraceRow> rows, int seed)
    {
        using (Py.GIL())
        {
            dynamic ensemble = Py.Import("sklearn.ensemble");
            np = Py.Import("numpy");
            dynamic numpy = np;
            dynamic classifier = ensemble.HistGradientBoostingClassifier(learning_rate: 0.08, max_iter: 160, max_leaf_nodes: 31, min_samples_leaf: 20, l2_regularization: 0.1, random_state: seed);
            var data = rows.ToArray();
            dynamic x = numpy.array(data.Select(r => r.Features).ToArray());
            dynamic y = numpy.array(data.Select(r => r.Label).ToArray());
            classifier.fit(x, y);
            model = (PyObject)classifier;
        }
    }

    public double[] Scores(double[][] rows)
    {
        using (Py.GIL())
        {
            dynamic numpy = np;
            dynamic m = model;
            dynamic scores = m.predict_proba(numpy.array(rows));
            string text = scores.tolist().ToString();
            using var doc = JsonDocument.Parse(text);
            return doc.RootElement.EnumerateArray().Select(x => x[1].GetDouble()).ToArray();
        }
    }

    public string Choose(string[] candidates, double[][] rows)
    {
        var scores = Scores(rows);
        int best = 0;
        for (int i = 1; i < scores.Length; i++)
            if (scores[i] > scores[best])
                best = i;
        return candidates[best];
    }

    public void Dispose()
    {
        using (Py.GIL())
        {
            model.Dispose();
            np.Dispose();
        }
    }
}

static double Median(IEnumerable<double> data)
{
    var a = data.OrderBy(x => x).ToArray();
    return a.Length % 2 == 1 ? a[a.Length / 2] : (a[a.Length / 2 - 1] + a[a.Length / 2]) / 2;
}

static double Geomean(IEnumerable<double> data) => Math.Exp(data.Average(Math.Log));
static double Top1(SklearnPolicy policy, TraceRow[] rows)
{
    var scores = policy.Scores(rows.Select(r => r.Features).ToArray());
    return rows.Select((r, i) => (r, i)).GroupBy(x => x.r.Node).Average(g => (double)g.OrderByDescending(x => scores[x.i]).First().r.Label);
}

// L'instance est l'unité du split ; le choix de baseline ne voit que le train.
void Experiment06(List<Instance> instances)
{
    var baselines = new List<SolveResult>();
    var trace = new List<TraceRow>();
    var names = new[]
    {
        "mrv",
        "max_degree",
        "dom_wdeg",
        "activity"
    };
    foreach (var instance in instances)
    {
        foreach (var heuristic in names)
        {
            var solver = new CspSolver(instance, heuristic, collect: heuristic == "dom_wdeg");
            baselines.Add(solver.Run());
            trace.AddRange(solver.Trace);
        }

        Console.WriteLine($"Baseline {instance.Id} : terminée");
    }

    var splits = new List<(string Name, HashSet<string> Train, HashSet<string> Test)>();
    var all = instances.Select(x => x.Id).ToHashSet();
    var grouped = instances.Where(c => int.Parse(c.Id[(c.Id.LastIndexOf('-') + 1)..]) < 8 || c.Family == "queens" && int.Parse(c.Id[(c.Id.LastIndexOf('-') + 1)..]) < 16).Select(c => c.Id).ToHashSet();
    splits.Add(("grouped_instance_split", grouped, all.Except(grouped).ToHashSet()));
    foreach (var f in new[]
    {
        "queens",
        "coloring",
        "latin"
    }

    )
    {
        var test = instances.Where(x => x.Family == f).Select(x => x.Id).ToHashSet();
        splits.Add(($"leave_{f}_out", all.Except(test).ToHashSet(), test));
    }

    var summaries = new List<object>();
    var evaluated = new List<object>();
    var evalCsv = new List<string>
    {
        "split,seed,instance_id,heuristic,solved,valid,nodes,seconds,inference_seconds"
    };
    foreach (var split in splits)
    {
        if (split.Train.Overlaps(split.Test))
            throw new InvalidOperationException("Fuite entre instances");
        string baseline = names.OrderBy(name => Median(baselines.Where(r => r.Heuristic == name && split.Train.Contains(r.Id)).Select(r => (double)r.Nodes))).ThenBy(x => x, StringComparer.Ordinal).First();
        foreach (int seed in new[]
        {
            11,
            23,
            42,
            71,
            101
        }

        )
        {
            using var policy = new SklearnPolicy(trace.Where(r => split.Train.Contains(r.Id)), seed);
            var results = new List<(SolveResult ML, SolveResult Base)>();
            foreach (var c in instances.Where(x => split.Test.Contains(x.Id)))
            {
                var learned = new CspSolver(c, "learned_policy", policy.Choose).Run();
                var baselineResult = baselines.Single(x => x.Id == c.Id && x.Heuristic == baseline);
                results.Add((learned, baselineResult));
                foreach (var r in new[]
                {
                    learned,
                    baselineResult
                }

                )
                    evalCsv.Add($"{split.Name},{seed},{r.Id},{r.Heuristic},{r.Solved},{r.Valid},{r.Nodes},{r.Seconds:R},{r.InferenceSeconds:R}");
                evaluated.Add(new { split = split.Name, seed, learned, baselineResult });
            }

            double top = Top1(policy, trace.Where(r => split.Test.Contains(r.Id)).ToArray());
            var nr = results.Select(x => (double)x.ML.Nodes / Math.Max(1, x.Base.Nodes)).ToArray();
            var tr = results.Select(x => x.ML.Seconds / Math.Max(1e-9, x.Base.Seconds)).ToArray();
            var summary = new
            {
                split = split.Name,
                seed,
                baseline,
                train = split.Train.Count,
                test = split.Test.Count,
                top1 = top,
                medianNodeRatio = Median(nr),
                geomeanNodeRatio = Geomean(nr),
                medianTimeRatio = Median(tr),
                nodeWins = results.Count(x => x.ML.Nodes < x.Base.Nodes),
                timeWins = results.Count(x => x.ML.Seconds < x.Base.Seconds),
                inferenceShare = Median(results.Select(x => x.ML.InferenceSeconds / Math.Max(x.ML.Seconds, 1e-9))),
                allValid = results.All(x => x.ML.Valid && x.Base.Valid),
                allSolved = results.All(x => x.ML.Solved && x.Base.Solved)
            };
            summaries.Add(summary);
            Console.WriteLine(JsonSerializer.Serialize(summary));
        }
    }

    var outdir = Path.Combine(Path.GetTempPath(), "coursia-frontieres06-csharp");
    Directory.CreateDirectory(outdir);
    File.WriteAllText(Path.Combine(outdir, "baseline_runs.json"), JsonSerializer.Serialize(baselines));
    var csv = new List<string>
    {
        "family,instance_id,heuristic,solved,valid,nodes,seconds,inference_seconds"
    };
    csv.AddRange(baselines.Select(r => $"{r.Family},{r.Id},{r.Heuristic},{r.Solved},{r.Valid},{r.Nodes},{r.Seconds:R},{r.InferenceSeconds:R}"));
    File.WriteAllLines(Path.Combine(outdir, "baseline_runs.csv"), csv);
    var traceCsv = new List<string>
    {
        "family,instance_id,node_id,candidate,domain_size,domain_ratio,degree,degree_ratio,weighted_degree,activity,dom_wdeg,progress,neighbor_domain_min,neighbor_domain_mean,neighbor_domain_max,label"
    };
    traceCsv.AddRange(trace.Select(r => $"{r.Family},{r.Id},{r.Node},{r.Candidate},{string.Join(",", r.Features.Select(x => x.ToString("R", System.Globalization.CultureInfo.InvariantCulture)))},{r.Label}"));
    File.WriteAllLines(Path.Combine(outdir, "oracle_trace.csv"), traceCsv);
    File.WriteAllText(Path.Combine(outdir, "oracle_trace.json"), JsonSerializer.Serialize(trace));
    File.WriteAllText(Path.Combine(outdir, "evaluation_runs.json"), JsonSerializer.Serialize(evaluated));
    File.WriteAllLines(Path.Combine(outdir, "evaluation_runs.csv"), evalCsv);
    File.WriteAllText(Path.Combine(outdir, "maturation_report.json"), JsonSerializer.Serialize(summaries));
    Console.WriteLine($"Instances : {instances.Count}, décisions : {trace.Select(x => x.Node).Distinct().Count()}, lignes de trace : {trace.Count}, résumés : {summaries.Count} ; artefacts JSON générés.");
}

Console.WriteLine("Politique sklearn, splits et métriques définis.");

Politique sklearn, splits et métriques définis.


### Exercice 2 — Choisir une baseline sans fuite

À partir de les résultats de baseline, choisissez l'heuristique de plus faible médiane de nœuds sur les **seules** instances d'entraînement d'un split.

**Questions d'interprétation :**

1. Pourquoi faut-il sélectionner une baseline différente pour chaque split ?
2. Comment départager deux heuristiques de médiane identique sans consulter le test ?
3. Pourquoi publier la règle de tie-break est-il nécessaire à la reproductibilité ?

**Étapes suggérées :** construire `train_ids`, filtrer les résultats C# avant tout regroupement, calculer médiane puis moyenne géométrique ou utiliser un ordre stable annoncé. Ajoutez une assertion garantissant que les `Id` test sont absents du tableau de sélection.

Le stub conserve `None` afin que l'exercice ne révèle pas la réponse et que le notebook reste exécutable.

In [6]:
// Exercice 2 : filtrer les baselines par les seuls identifiants du train.
string selectedBaseline = null; // TODO étudiant
Console.WriteLine(selectedBaseline ?? "Exercice à compléter");

Exercice à compléter


### Imiter un oracle heuristique ne crée pas une oracle contrefactuelle

Le classifieur apprend la décision prise par `dom/wdeg`, pas la variable qui minimiserait réellement le sous-arbre. Même une imitation top-1 parfaite reproduirait donc les qualités et limites de son professeur. Une imitation imparfaite peut parfois gagner par hasard en s'écartant de l'oracle ; ce cas doit être mesuré dans le solveur plutôt qu'interprété comme une amélioration apprise.

`choose_train_baseline` évite une autre fuite : la baseline de comparaison est sélectionnée sur les seules instances train. La choisir après lecture du test donnerait au comparateur classique un avantage rétrospectif et rendrait le duel aussi contaminé que le modèle.

## 5. Protocole complet — du candidat à la trajectoire

Chaque split est répété avec cinq graines du modèle. Les mêmes CSP held-out sont présentés à la politique apprise et à la baseline sélectionnée. Pour chaque paire, le protocole conserve :

- validation et résolution complète ;
- ratio de nœuds $n_{ML}/n_{base}$ ;
- ratio de temps $t_{ML}/t_{base}$ ;
- victoires strictes en nœuds et en temps ;
- part $t_{inference}/t_{ML}$ ;
- top-1 par nœud sur les traces held-out de l'oracle.

La moyenne géométrique des ratios limite l'effet d'échelle entre instances, tandis que la médiane décrit une instance centrale. Une victoire stricte exige `<`, pas `≤` : égaler MRV en nœuds tout en ajoutant de l'inférence n'est pas une victoire opérationnelle.

In [7]:
// Le banc complet conserve quatre baselines, quatre splits et cinq graines.
Console.WriteLine("Protocole : 36 instances, limite 20 000 nœuds, sélection de baseline sur train uniquement.");

Protocole : 36 instances, limite 20 000 nœuds, sélection de baseline sur train uniquement.


## 6. Exécution fraîche — matérialiser le coût de chaque claim

La cellule suivante régénère les CSV et le rapport JSON dans un répertoire temporaire propre au port C#. Elle résout d'abord les 36 instances avec quatre heuristiques classiques, collecte les lignes candidates aux nœuds visités par `dom/wdeg`, puis entraîne et évalue 20 combinaisons split × graine.

Le coût de cette exécution est partie intégrante de l'expérience : une politique de branchement n'est pas évaluée hors du solveur. Les temps absolus restent dépendants de la machine et de la charge ; les ratios comparent deux méthodes dans le même run, sur la même instance. Même ces ratios ne sont pas universels, car les CSP sont petits et l'overhead Python occupe une place plus grande que dans un solveur compilé.

Les artefacts du port sont générés à chaque exécution depuis les fixtures d’entrées ; les cellules suivantes recalculent les synthèses. Les fichiers historiques du carnet Python restent distincts. Toute divergence entre prose et sortie doit être corrigée après le rerun, jamais masquée par une ancienne capture.

In [8]:
// Mesures fraîches : résoudre puis entraîner et réinjecter le modèle.
Experiment06(instances);

Baseline queens-8 : terminée


Baseline queens-9 : terminée


Baseline queens-10 : terminée


Baseline queens-11 : terminée


Baseline queens-12 : terminée


Baseline queens-13 : terminée


... (47 outputs left out)

{"split":"leave_latin_out","seed":42,"baseline":"mrv","train":24,"test":12,"top1":0.5792349726775956,"medianNodeRatio":1,"geomeanNodeRatio":1.0072614368503952,"medianTimeRatio":6.098842121089882,"nodeWins":0,"timeWins":0,"inferenceShare":0.9217702818058364,"allValid":true,"allSolved":true}


{"split":"leave_latin_out","seed":71,"baseline":"mrv","train":24,"test":12,"top1":0.5792349726775956,"medianNodeRatio":1,"geomeanNodeRatio":1.0072614368503952,"medianTimeRatio":4.7026113633474065,"nodeWins":0,"timeWins":0,"inferenceShare":0.9093544451608199,"allValid":true,"allSolved":true}


{"split":"leave_latin_out","seed":101,"baseline":"mrv","train":24,"test":12,"top1":0.5792349726775956,"medianNodeRatio":1,"geomeanNodeRatio":1.0072614368503952,"medianTimeRatio":12.214312255506872,"nodeWins":0,"timeWins":0,"inferenceShare":0.9177371552343954,"allValid":true,"allSolved":true}


Instances : 36, décisions : 794, lignes de trace : 11461, résumés : 20 ; artefacts JSON générés.


### Lire les mesures

Les tableaux ci-dessus sont calculés pour chaque split et graine ; leur interprétation dépend des observations de cette exécution.

**Contrôle indépendant de parité.** Les voisins Python sont des `set` dont l’ordre dépend du hachage : l’arrêt anticipé d’AC-3 sur domaine vide peut changer la feature `activity`. Un contrôle séparé du run ci-dessus a fait varier `PYTHONHASHSEED` et observé cette variation. Lorsque Python utilise le même ordre d’arcs et de voisins que le C#, les **11 461 lignes** de trace, leurs candidats, labels et onze features concordent à une tolérance de **1e−12**. Cette vérification ne remplace pas les mesures fraîches du carnet ; les temps restent propres à leur exécution.


### Exercice 3 — Concevoir une règle d'abstention

Proposez une politique hybride : employer le modèle seulement lorsque son score dépasse un seuil et revenir à `dom/wdeg` sinon. Mesurez nœuds **et** temps mur.

**Questions d'interprétation :**

1. Une probabilité élevée de classe 1 est-elle une confiance calibrée entre candidats d'un même nœud ?
2. Quel seuil choisissez-vous sans consulter les instances test ?
3. Une abstention fréquente peut-elle rester utile si elle paie tout de même le coût de `predict_proba` ?

**Étapes suggérées :** calibrer le seuil sur une validation issue du train, retourner `None` sous le seuil, faire appliquer alors `dom/wdeg`, puis rapporter taux d'abstention, top-1 conditionnel, nœuds et temps total. Comparez aussi un critère de marge entre les deux meilleurs scores.

Le stub est sûr : il retourne `None`, ce qui représente précisément la décision d'abstention à traiter.

In [9]:
// Exercice 3 : renvoyer null pour s'abstenir sous un seuil.
string AbstainingPolicy(string[] candidates, double[][] rows, double threshold = .9)
{ /* TODO étudiant */
    return null;
}

Console.WriteLine(AbstainingPolicy(Array.Empty<string>(), Array.Empty<double[]>()) ?? "Exercice à compléter");

Exercice à compléter


### Comparer les splits

Les synthèses suivantes regroupent les résultats frais par protocole et par graine.

In [10]:
// Relecture des mesures produites au cours de cette exécution.
var report = JsonDocument.Parse(File.ReadAllText(Path.Combine(Path.GetTempPath(), "coursia-frontieres06-csharp", "maturation_report.json"))).RootElement.EnumerateArray().ToArray();
foreach (var group in report.GroupBy(x => x.GetProperty("split").GetString()))
    Console.WriteLine($"{group.Key} : graines={group.Count()}, top1=[{group.Min(x => x.GetProperty("top1").GetDouble()):F3}, {group.Max(x => x.GetProperty("top1").GetDouble()):F3}], ratio nœuds médian=[{group.Min(x => x.GetProperty("medianNodeRatio").GetDouble()):F2}, {group.Max(x => x.GetProperty("medianNodeRatio").GetDouble()):F2}]");

grouped_instance_split : graines=5, top1=[0.580, 0.580], ratio nœuds médian=[1.00, 1.00]


leave_queens_out : graines=5, top1=[0.621, 0.643], ratio nœuds médian=[1.42, 2.82]


leave_coloring_out : graines=5, top1=[0.411, 0.411], ratio nœuds médian=[1.00, 1.00]


leave_latin_out : graines=5, top1=[0.579, 0.579], ratio nœuds médian=[1.00, 1.00]


### Stabilité et généralisation

La stabilité d’un top-1 ne garantit ni transfert de famille ni gain de temps : comparer ratios de nœuds et de temps, en gardant les solutions valides.

In [11]:
// L'inférence est incluse dans le temps total.
foreach (var group in report.GroupBy(x => x.GetProperty("split").GetString()))
    Console.WriteLine($"{group.Key} : part d'inférence médiane par graine = {string.Join(", ", group.Select(x => x.GetProperty("inferenceShare").GetDouble().ToString("F3")))} ; victoires temps = {group.Sum(x => x.GetProperty("timeWins").GetInt32())}");

grouped_instance_split : part d'inférence médiane par graine = 0.908, 0.934, 0.911, 0.900, 0.876 ; victoires temps = 2


leave_queens_out : part d'inférence médiane par graine = 0.775, 0.719, 0.675, 0.743, 0.771 ; victoires temps = 6


leave_coloring_out : part d'inférence médiane par graine = 0.953, 0.962, 0.965, 0.964, 0.958 ; victoires temps = 0


leave_latin_out : part d'inférence médiane par graine = 0.924, 0.915, 0.922, 0.909, 0.918 ; victoires temps = 0


### Exercice 4 — Calculer le seuil d'amortissement

Supposons que la baseline coûte $c_b$ par nœud et que la politique apprise ajoute $c_{ML}$ de scoring. Dérivez la réduction minimale de nœuds nécessaire pour égaler le temps de la baseline :

$$n_{ML}(c_b+c_{ML})\le n_b c_b.$$

**Questions d'interprétation :**

1. Quel ratio $n_{ML}/n_b$ faut-il atteindre lorsque l'inférence représente 85 % du temps ML ?
2. Pourquoi ce seuil dépend-il de l'implémentation et de la taille du CSP ?
3. Batcher plusieurs nœuds est-il compatible avec un backtracking strictement séquentiel ?

**Étapes suggérées :** isoler $n_{ML}/n_b$, estimer $c_{ML}/c_b$ depuis les timings du run, puis comparer ce seuil aux ratios observés. Effectuez une analyse de sensibilité plutôt que de traiter un timing unique comme une constante.

Cet exercice transforme le résultat négatif en condition d'ingénierie : combien d'arbre faut-il réellement économiser pour payer le modèle ?

In [12]:
// Exercice 4 : dériver le ratio de nœuds permettant d'amortir l'inférence.
double? BreakEvenNodeRatio(double inferenceShare)
{ /* TODO étudiant */
    return null;
}

Console.WriteLine(BreakEvenNodeRatio(.85)?.ToString() ?? "Exercice à compléter");

Exercice à compléter


## À retenir

Le split suit l’instance. L’imitation locale, l’effet sur les nœuds et le temps total mesurent des propriétés distinctes. Un validateur contrôle chaque affectation et les cinq graines mesurent l’instabilité de l’entraînement, pas cinq jeux de test indépendants.

## Vérifiez votre compréhension

Pourquoi faut-il choisir la baseline uniquement sur les données d’entraînement ? Pourquoi un gain en nœuds peut-il coûter du temps ?

<details><summary>Éléments de réponse</summary>

La sélection sur le test introduirait une fuite. Le scoring du modèle s’ajoute au coût de chaque nœud et peut dépasser l’économie de recherche.
</details>

## Pour aller plus loin

- Boussemart et al. (2004), *Boosting systematic search by weighting constraints*.
- [Index Partie 5](README.md).